# Large File Copy Benchmark
Benchmarks S3 → UC Volume throughput using `dbutils.fs.cp` + `ThreadPool`. Horizontally scalable in increments of 32 workers by running multiple job instances.

| Role | Condition | Action |
|------|-----------|--------|
| **Coordinator** | `instance_num == 0` | Submits `total_instances` worker jobs, waits, displays aggregate results |
| **Worker** | `instance_num > 0` | Copies assigned pages, logs metrics to MLflow |

**Page assignment**: interleaved — worker `i` handles pages where `page_num % total_instances == i - 1`  
**Threads per worker**: 32 (fixed)  
**Total threads**: `32 × total_instances`


Thoughts on improvements

## TODOs:
1. Add config to pull in AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY value from databricks secrets api. The configs would name the scope and the keys to the databricks secrets
2. If secrets are found, generate a S3 URL that packs the keys in using the correctly escape characters
3. Find out why each worker has 75s startup time before copying
0. Create a status table that parallels manifest table (put this in a setup notebook) that is used to track file level metrics and copy status.
0. Change MANIFEST_TABLE config into a MANIFEST_QUERY confg.
0. Make job idempotent (with help of MANIFEST_QUERY and status table).
0. Assume management of MANIFEST_TABLE and file status tracking table are performed outside this notebook.
0. MANIFEST_QUERY should select a target_path, setup will establish this target path based on a target root folder.
0. Change print() to python logging
0. Add a config for the worker instance type. Default to "serverless", add option for Azure 8 core cpu/network optimized instance type.
0. Add retry logic, retry once & log, fail & log and move on (upto 3) then fail job.
0. Add upfront tests for existance of credentials, MANIFEST_TABLE, target path, source path into the coordinator run.
0. Make sure there are no bottlenecks that would prevent this from handling 1 billion files over the course of 4 weeks and multiple crashes and re-starts.
0. Make workers per instance a config, default to 32.

In [ ]:
import yaml
import math

with open("config.yaml", "r") as _f:
    _cfg = yaml.safe_load(_f)

COMPUTE_MODE      = _cfg["COMPUTE_MODE"]
DEDICATED_CLUSTER = _cfg.get("DEDICATED_CLUSTER", {})

TOTAL_FILES       = _cfg["TOTAL_FILES"]
PAGE_SIZE         = _cfg["PAGE_SIZE"]
NUM_WORKERS       = _cfg["NUM_WORKERS"]
MANIFEST_TABLE    = _cfg["MANIFEST_TABLE"]
DEST_VOLUME       = _cfg["DEST_VOLUME"]
MLFLOW_EXPERIMENT = _cfg["MLFLOW_EXPERIMENT"]

# Discover this notebook's workspace path at runtime so the coordinator can
# submit worker jobs that re-run the same notebook.
from dbruntime.databricks_repl_context import get_context
NOTEBOOK_PATH     = get_context().notebookPath

SECRETS_SCOPE                 = _cfg["SECRETS_SCOPE"]
SECRETS_KEY_ACCESS_KEY_ID     = _cfg["SECRETS_KEY_ACCESS_KEY_ID"]
SECRETS_KEY_SECRET_ACCESS_KEY = _cfg["SECRETS_KEY_SECRET_ACCESS_KEY"]
REGION            = _cfg["REGION"]
ENDPOINT_URL      = _cfg["ENDPOINT_URL"]
SCHEME            = _cfg["SCHEME"]

In [0]:
# Parameters — set automatically when run as a serverless job;
# fill in manually for interactive use.
dbutils.widgets.text("instance_num",    "0", "Instance # (0 = coordinator)")
dbutils.widgets.text("total_instances", "1", "Total Worker Instances")
dbutils.widgets.text("run_guid",        "",  "Run GUID (auto-set by coordinator; leave blank)")
dbutils.widgets.text("parent_run_id",   "",  "Parent MLflow Run ID (internal)")
dbutils.widgets.text("worker_pool_id",  "387189596215963", "Worker Instance Pool ID")

instance_num                = int(dbutils.widgets.get("instance_num"))
total_instances             = int(dbutils.widgets.get("total_instances"))
run_guid_param              = dbutils.widgets.get("run_guid")
mlflow_parent_run_id_param  = dbutils.widgets.get("parent_run_id")
worker_pool_id_param        = dbutils.widgets.get("worker_pool_id")

role = "COORDINATOR" if instance_num == 0 else f"WORKER (instance {instance_num} of {total_instances})"
print(f"Role:          {role}")
print(f"Total threads: {32 * total_instances} ({total_instances} × 32)")


TOTAL_PAGES = math.ceil(TOTAL_FILES / PAGE_SIZE)
print(f"TOTAL_FILES={TOTAL_FILES}  PAGE_SIZE={PAGE_SIZE}  TOTAL_PAGES={TOTAL_PAGES}")
print(f"Total copy threads: {NUM_WORKERS * total_instances}")

print(f"Endpoint URL:       {ENDPOINT_URL}")
print(f"MANIFEST_TABLE=     {MANIFEST_TABLE}")
print(f"DEST_VOLUME=        {DEST_VOLUME}")
print(f"MLFLOW_EXPERIMENT=  {MLFLOW_EXPERIMENT}")

In [0]:
# Set Hadoop/Spark settings to add AWS credentials
# Requires dedicated cluster

spark.conf.set("fs.s3a.access.key", dbutils.secrets.get(scope=SECRETS_SCOPE, key=SECRETS_KEY_ACCESS_KEY_ID))
spark.conf.set("fs.s3a.secret.key", dbutils.secrets.get(scope=SECRETS_SCOPE, key=SECRETS_KEY_SECRET_ACCESS_KEY))
spark.conf.set("spark.hadoop.fs.s3a.path.style.access", "true") 
spark.conf.set("fs.s3a.endpoint", f"s3.{REGION}.amazonaws.com")
spark.conf.set("fs.s3a.endpoint_url", f"https://s3.{REGION}.amazonaws.com")
spark.conf.set("fs.s3a.connection.ssl.enabled", "true")

In [0]:
import time
import uuid
import threading
from typing import List, Dict

import psutil
import mlflow
import pyspark.sql.functions as F
from concurrent.futures import ThreadPoolExecutor, as_completed

# Validate manifest exists and contains the configured benchmark slice
try:
    manifest_df = spark.table(MANIFEST_TABLE)
    manifest_scope_df = manifest_df.filter(F.col("file_id") < TOTAL_FILES)
    scope_count = manifest_scope_df.count()
    if scope_count == 0:
        raise ValueError("Configured benchmark slice is empty")
    if scope_count < TOTAL_FILES:
        raise ValueError(
            f"Manifest only contains {scope_count} rows for file_id < {TOTAL_FILES}; expected {TOTAL_FILES}"
        )
    print(f"✓ Manifest: {MANIFEST_TABLE} ({scope_count} benchmark rows)")
except Exception as e:
    raise RuntimeError(
        f"Manifest table not found or does not contain the configured benchmark slice.\n{e}"
    )

mlflow.set_experiment(MLFLOW_EXPERIMENT)
print(f"✓ MLflow experiment: {MLFLOW_EXPERIMENT}")

In [0]:
# ═══════════════════════════════════════════════════════════════════════════════
# COORDINATOR  (instance_num == 0)
# Creates a parent MLflow run, submits total_instances serverless worker jobs
# that log nested child runs, waits for completion, then logs aggregate stats.
# ═══════════════════════════════════════════════════════════════════════════════
if instance_num != 0:
    print("Skipping — worker mode")
else:
    from databricks.sdk import WorkspaceClient
    from databricks.sdk.service.jobs import (
        SubmitTask, NotebookTask, RunLifeCycleState, RunResultState
    )
    from databricks.sdk.service.compute import ClusterSpec, AutoScale

    run_guid = str(uuid.uuid4())[:8]
    benchmark_scope_df = spark.table(MANIFEST_TABLE).filter(F.col("file_id") < TOTAL_FILES)
    benchmark_scope_stats = benchmark_scope_df.agg(
        F.count("*").alias("planned_files"),
        F.min("size_bytes").alias("planned_min_file_size_bytes"),
        F.max("size_bytes").alias("planned_max_file_size_bytes"),
        F.avg("size_bytes").alias("planned_avg_file_size_bytes"),
        F.sum("size_bytes").alias("planned_total_bytes"),
    ).collect()[0].asDict()

    coordinator_t0 = time.time()
    w = WorkspaceClient()

    with mlflow.start_run(run_name=f"parent_{run_guid}") as parent_run:
        mlflow_parent_run_id = parent_run.info.run_id
        print(f"Coordinator  run_guid={run_guid}  mlflow_parent_run_id={mlflow_parent_run_id}")
        print(f"Launching {total_instances} worker(s)…")

        mlflow.set_tags({
            "phase": "scaled_benchmark",
            "role": "parent",
            "run_guid": run_guid,
            "compute_type": COMPUTE_MODE,
        })
        mlflow.log_params({
            "run_guid": run_guid,
            "instance_num": instance_num,
            "total_instances": total_instances,
            "num_workers_per_instance": NUM_WORKERS,
            "page_size": PAGE_SIZE,
            "total_pages": TOTAL_PAGES,
            "total_files_requested": TOTAL_FILES,
            "manifest_table": MANIFEST_TABLE,
            "dest_volume": DEST_VOLUME,
            "notebook_path": NOTEBOOK_PATH,
            "worker_pool_id": worker_pool_id_param or "",
        })
        mlflow.log_metrics({
            "planned_files": int(benchmark_scope_stats["planned_files"] or 0),
            "planned_total_bytes": float(benchmark_scope_stats["planned_total_bytes"] or 0),
            "planned_min_file_size_bytes": float(benchmark_scope_stats["planned_min_file_size_bytes"] or 0),
            "planned_max_file_size_bytes": float(benchmark_scope_stats["planned_max_file_size_bytes"] or 0),
            "planned_avg_file_size_bytes": float(benchmark_scope_stats["planned_avg_file_size_bytes"] or 0),
        })

        # Build cluster spec based on COMPUTE_MODE
        if COMPUTE_MODE == "dedicated":
            _cluster_spec = {
                "spark_version": DEDICATED_CLUSTER["spark_version"],
                "num_workers": 0,
                "data_security_mode": DEDICATED_CLUSTER["data_security_mode"],
                "spark_conf": {
                    "spark.databricks.cluster.profile": "singleNode",
                    "spark.master": "local[*]",
                },
                "custom_tags": {
                    "ResourceClass": "SingleNode",
                },
            }
            if worker_pool_id_param:
                _cluster_spec["instance_pool_id"] = worker_pool_id_param
                print(f"  Compute: dedicated single-node (pool={worker_pool_id_param})")
            else:
                _cluster_spec["node_type_id"] = DEDICATED_CLUSTER["node_type_id"]
                print(f"  Compute: dedicated single-node ({DEDICATED_CLUSTER['node_type_id']})")
            _new_cluster = ClusterSpec.from_dict(_cluster_spec)
        else:
            _new_cluster = None
            print("  Compute: serverless")

        # Submit one job per worker instance
        job_run_ids = []
        for i in range(1, total_instances + 1):
            task = SubmitTask(
                task_key=f"big_file_mover_bench_worker_{i}",
                notebook_task=NotebookTask(
                    notebook_path=NOTEBOOK_PATH,
                    base_parameters={
                        "instance_num": str(i),
                        "total_instances": str(total_instances),
                        "run_guid": run_guid,
                        "parent_run_id": mlflow_parent_run_id,
                    },
                ),
            )
            if _new_cluster:
                task.new_cluster = _new_cluster

            resp = w.jobs.submit(
                run_name=f"big_file_mover_bench_{run_guid}_w{i}",
                tasks=[task],
            )
            job_run_ids.append(resp.run_id)
            print(f"  Submitted worker {i}: job_run_id={resp.run_id}")

        # Poll until every worker reaches a terminal state
        print(f"\nWaiting for {len(job_run_ids)} worker(s)…")
        failed = []
        for job_run_id in job_run_ids:
            while True:
                state = w.jobs.get_run(run_id=job_run_id).state
                lc = state.life_cycle_state
                if lc in (
                    RunLifeCycleState.TERMINATED,
                    RunLifeCycleState.SKIPPED,
                    RunLifeCycleState.INTERNAL_ERROR,
                ):
                    ok = state.result_state == RunResultState.SUCCESS
                    sym = "✓" if ok else "✗"
                    print(f"  {sym} job_run_id={job_run_id}: {state.result_state}  {state.state_message or ''}")
                    if not ok:
                        failed.append(job_run_id)
                    break
                time.sleep(10)

        child_runs = mlflow.search_runs(
            experiment_ids=[mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT).experiment_id],
            filter_string=(
                f"tags.run_guid = '{run_guid}' "
                f"AND tags.phase = 'scaled_benchmark' "
                f"AND tags.role = 'worker'"
            ),
        )

        parent_wall_time_sec = time.time() - coordinator_t0
        worker_run_count = len(child_runs)
        files_copied = int(child_runs["metrics.files_copied"].sum()) if not child_runs.empty else 0
        total_bytes_copied = float(child_runs["metrics.total_bytes"].sum()) if not child_runs.empty else 0.0
        max_worker_wall_time_sec = float(child_runs["metrics.wall_time_sec"].max()) if not child_runs.empty else 0.0
        total_errors = int(child_runs["metrics.error_count"].sum()) if not child_runs.empty else 0
        aggregate_throughput_mbps = (
            (total_bytes_copied / (1024 ** 2)) / max_worker_wall_time_sec
            if max_worker_wall_time_sec > 0 else 0.0
        )
        min_file_size_bytes = float(child_runs["metrics.min_file_size_bytes"].min()) if files_copied > 0 else 0.0
        max_file_size_bytes = float(child_runs["metrics.max_file_size_bytes"].max()) if files_copied > 0 else 0.0
        avg_file_size_bytes = (total_bytes_copied / files_copied) if files_copied > 0 else 0.0

        mlflow.log_metrics({
            "worker_run_count": worker_run_count,
            "failed_worker_run_count": len(failed),
            "files_copied": files_copied,
            "total_bytes_copied": total_bytes_copied,
            "parent_wall_time_sec": round(parent_wall_time_sec, 2),
            "max_worker_wall_time_sec": round(max_worker_wall_time_sec, 2),
            "aggregate_throughput_mbps": round(aggregate_throughput_mbps, 2),
            "total_error_count": total_errors,
            "min_file_size_bytes": round(min_file_size_bytes, 2),
            "max_file_size_bytes": round(max_file_size_bytes, 2),
            "avg_file_size_bytes": round(avg_file_size_bytes, 2),
        })

        if failed:
            print(f"\n⚠ {len(failed)} worker(s) failed: {failed}")
        else:
            print(f"\nAll {len(job_run_ids)} worker(s) completed successfully.")

        print(
            f"Aggregate: {files_copied} files copied · "
            f"{total_bytes_copied / (1024 ** 3):.2f} GB · "
            f"{aggregate_throughput_mbps:.1f} MB/s"
        )

    MLFLOW_PARENT_RUN_ID = mlflow_parent_run_id

In [0]:
# ═══════════════════════════════════════════════════════════════════════════════
# WORKER  (instance_num > 0)
# Copies the interleaved pages assigned to this instance, then logs metrics
# as a nested MLflow child run under the coordinator's parent run.
# ═══════════════════════════════════════════════════════════════════════════════
if instance_num == 0:
    print("Skipping — coordinator mode")
else:
    run_guid = run_guid_param
    if not run_guid:
        raise ValueError("run_guid is required for worker instances (set automatically by coordinator).")
    if not mlflow_parent_run_id_param:
        raise ValueError("parent_run_id is required for worker instances so MLflow can log nested child runs.")

    # ── Discover own job run ID from notebook context ──────────────────────────────
    from dbruntime import databricks_repl_context
    job_run_id = databricks_repl_context.get_context().currentRunId

    # ── Interleaved page assignment ────────────────────────────────────────────
    # Worker i (1-based) owns pages where: page_num % total_instances == i - 1
    assigned_pages = [
        p for p in range(TOTAL_PAGES)
        if p % total_instances == (instance_num - 1)
    ]
    print(f"Worker {instance_num}/{total_instances}: pages {assigned_pages}")

    # ── Load files from manifest ───────────────────────────────────────────────
    if assigned_pages:
        my_files = [
            row.asDict()
            for row in spark.table(MANIFEST_TABLE)
                            .filter(F.col("file_id") < TOTAL_FILES)
                            .filter(F.col("page_num").isin(assigned_pages))
                            .orderBy("file_id")
                            .collect()
        ]
    else:
        my_files = []

    assigned_file_sizes = [f["size_bytes"] for f in my_files]
    total_bytes_assigned = sum(assigned_file_sizes)
    assigned_min_file_size = min(assigned_file_sizes) if assigned_file_sizes else 0
    assigned_max_file_size = max(assigned_file_sizes) if assigned_file_sizes else 0
    assigned_avg_file_size = (total_bytes_assigned / len(assigned_file_sizes)) if assigned_file_sizes else 0
    print(f"  {len(my_files)} files · {total_bytes_assigned / (1024**3):.2f} GB")

    # ── Destination directory ──────────────────────────────────────────────────
    output_dir = f"{DEST_VOLUME}/{run_guid}"
    dbutils.fs.mkdirs(output_dir)

    # ── Background system monitor ──────────────────────────────────────────────
    class SystemMonitor:
        def __init__(self, interval: float = 1.0):
            self.interval = interval
            self.cpu_samples: List[float] = []
            self.net_samples: List[float] = []
            self._stop = threading.Event()
            self._thread = threading.Thread(target=self._run, daemon=True)

        def start(self):
            self._thread.start()

        def stop(self):
            self._stop.set()
            self._thread.join()

        def _run(self):
            prev = psutil.net_io_counters()
            while not self._stop.is_set():
                time.sleep(self.interval)
                cur = psutil.net_io_counters()
                self.cpu_samples.append(psutil.cpu_percent())
                self.net_samples.append(cur.bytes_recv - prev.bytes_recv)
                prev = cur

    # ── Copy function: dbutils.fs.cp with one retry ────────────────────────────
    def _cp_one(f: Dict):
        #TODO: Get aws secrets from SCOPE = "dm-aws-credentials" 
        src = f"{SCHEME}{f['source_bucket']}/{f['source_key']}"
        dst = f"{output_dir}/{f['dest_filename']}"
        t0 = time.time()
        try:
            dbutils.fs.cp(src, dst)
            return (time.time() - t0, f["size_bytes"], None)
        except Exception:
            try:
                time.sleep(1)
                dbutils.fs.cp(src, dst)
                return (time.time() - t0, f["size_bytes"], None)
            except Exception as e2:
                return (time.time() - t0, 0, str(e2))

    # ── Execute ────────────────────────────────────────────────────────────────
    monitor = SystemMonitor()
    monitor.start()
    t0 = time.time()
    per_file_times: List[float] = []
    per_file_bytes: List[int] = []
    errors = 0

    with ThreadPoolExecutor(max_workers=NUM_WORKERS) as pool:
        futures = [pool.submit(_cp_one, f) for f in my_files]
        for future in as_completed(futures):
            elapsed, nbytes, err = future.result()
            per_file_times.append(elapsed)
            per_file_bytes.append(nbytes)
            if err:
                print(f"ERROR: {err}")
                errors += 1

    wall_time = time.time() - t0
    monitor.stop()

    # ── Derive metrics ─────────────────────────────────────────────────────────
    copied_file_sizes = [b for b in per_file_bytes if b > 0]
    files_copied = len(copied_file_sizes)
    total_bytes = sum(copied_file_sizes)
    gb = total_bytes / (1024 ** 3)
    tput_mbps = (total_bytes / (1024 ** 2)) / wall_time if wall_time > 0 else 0.0
    cpu_avg = sum(monitor.cpu_samples) / len(monitor.cpu_samples) if monitor.cpu_samples else 0.0
    net_peak = max(monitor.net_samples) / (1024 ** 2) if monitor.net_samples else 0.0
    rates = [
        b / (1024 ** 2) / t
        for b, t in zip(per_file_bytes, per_file_times)
        if t > 0 and b > 0
    ]
    successful_times = [t for t, b in zip(per_file_times, per_file_bytes) if b > 0]
    avg_file_t = sum(successful_times) / len(successful_times) if successful_times else 0.0
    min_file_size_bytes = min(copied_file_sizes) if copied_file_sizes else 0
    max_file_size_bytes = max(copied_file_sizes) if copied_file_sizes else 0
    avg_file_size_bytes = (total_bytes / files_copied) if files_copied > 0 else 0.0

    print(
        f"\nWorker {instance_num} done: "
        f"{files_copied} files copied · {tput_mbps:.1f} MB/s · {wall_time:.1f}s · {errors} error(s)"
    )

    # ── Log to MLflow ──────────────────────────────────────────────────────────
    with mlflow.start_run(
        run_name=f"w{instance_num}_of_{total_instances}_{run_guid}",
        nested=True,
        parent_run_id=mlflow_parent_run_id_param,
    ):
        mlflow.set_tags({
            "phase": "scaled_benchmark",
            "role": "worker",
            "run_guid": run_guid,
            "instance_num": str(instance_num),
            "total_instances": str(total_instances),
            "compute_type": COMPUTE_MODE,
            "mlflow.parentRunId": mlflow_parent_run_id_param,
        })
        mlflow.log_params({
            "instance_num": instance_num,
            "total_instances": total_instances,
            "num_files_assigned": len(my_files),
            "num_workers": NUM_WORKERS,
            "pages": str(assigned_pages),
            "run_guid": run_guid,
            "job_run_id": job_run_id,
        })
        mlflow.log_metrics({
            "files_copied": files_copied,
            "total_bytes": total_bytes,
            "wall_time_sec": round(wall_time, 2),
            "throughput_mbps": round(tput_mbps, 2),
            "error_count": errors,
            "cpu_utilization_pct": round(cpu_avg, 1),
            "network_peak_mbps": round(net_peak, 2),
            "min_file_throughput_mbps": round(min(rates), 2) if rates else 0,
            "max_file_throughput_mbps": round(max(rates), 2) if rates else 0,
            "avg_file_time_sec": round(avg_file_t, 2),
            "min_file_size_bytes": float(min_file_size_bytes),
            "max_file_size_bytes": float(max_file_size_bytes),
            "avg_file_size_bytes": round(avg_file_size_bytes, 2),
            "assigned_min_file_size_bytes": float(assigned_min_file_size),
            "assigned_max_file_size_bytes": float(assigned_max_file_size),
            "assigned_avg_file_size_bytes": round(assigned_avg_file_size, 2),
        })

In [0]:
# ═══════════════════════════════════════════════════════════════════════════════
# AGGREGATE RESULTS  (coordinator only)
# Queries MLflow by run_guid, prints parent-run aggregate metrics, and displays
# per-worker child-run results.
# ═══════════════════════════════════════════════════════════════════════════════
if instance_num != 0:
    print("Skipping — worker mode")
else:
    selected_run_guid = locals().get("run_guid", None) or run_guid_param
    if not selected_run_guid:
        raise ValueError("No run_guid is available. Run the coordinator cell first or set the run_guid widget.")

    exp = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT)
    parent_runs = mlflow.search_runs(
        experiment_ids=[exp.experiment_id],
        filter_string=(
            f"tags.run_guid = '{selected_run_guid}' "
            f"AND tags.phase = 'scaled_benchmark' "
            f"AND tags.role = 'parent'"
        ),
    )
    worker_runs = mlflow.search_runs(
        experiment_ids=[exp.experiment_id],
        filter_string=(
            f"tags.run_guid = '{selected_run_guid}' "
            f"AND tags.phase = 'scaled_benchmark' "
            f"AND tags.role = 'worker'"
        ),
    )

    if worker_runs.empty and parent_runs.empty:
        print(f"No MLflow runs found for run_guid={selected_run_guid}. Workers may still be starting.")
    else:
        parent_row = parent_runs.iloc[0] if not parent_runs.empty else None

        if parent_row is not None:
            print(f"\n{'═' * 68}")
            print(f"  BENCHMARK RESULTS   run_guid = {selected_run_guid}")
            print(f"{'═' * 68}")
            print(f"  Parent run id:         {parent_row['run_id']}")
            print(f"  Worker instances:      {int(parent_row.get('params.total_instances', total_instances))}")
            print(f"  Total threads:         {int(parent_row.get('params.num_workers_per_instance', NUM_WORKERS)) * int(parent_row.get('params.total_instances', total_instances))}")
            print(f"  Planned files:         {int(parent_row.get('metrics.planned_files', 0))}")
            print(f"  Files copied:          {int(parent_row.get('metrics.files_copied', 0))}")
            print(f"  Planned total GB:      {float(parent_row.get('metrics.planned_total_bytes', 0)) / (1024 ** 3):.2f}")
            print(f"  Copied total GB:       {float(parent_row.get('metrics.total_bytes_copied', 0)) / (1024 ** 3):.2f}")
            print(f"  Parent wall time:      {float(parent_row.get('metrics.parent_wall_time_sec', 0)):.1f} s")
            print(f"  Max worker wall time:  {float(parent_row.get('metrics.max_worker_wall_time_sec', 0)):.1f} s")
            print(f"  Aggregate throughput:  {float(parent_row.get('metrics.aggregate_throughput_mbps', 0)):.1f} MB/s")
            print(f"  Total errors:          {int(parent_row.get('metrics.total_error_count', 0))}")
            print(f"  Min file size:         {float(parent_row.get('metrics.min_file_size_bytes', 0)) / (1024 ** 2):.2f} MB")
            print(f"  Max file size:         {float(parent_row.get('metrics.max_file_size_bytes', 0)) / (1024 ** 2):.2f} MB")
            print(f"  Avg file size:         {float(parent_row.get('metrics.avg_file_size_bytes', 0)) / (1024 ** 2):.2f} MB")
            print(f"{'═' * 68}\n")
        else:
            print(f"No parent run found yet for run_guid={selected_run_guid}; showing worker runs only.\n")

        if worker_runs.empty:
            print("No worker runs found yet.")
        else:
            col_map = {
                "params.instance_num": "Worker",
                "params.num_files_assigned": "Files Assigned",
                "metrics.files_copied": "Files Copied",
                "metrics.wall_time_sec": "Wall Time (s)",
                "metrics.throughput_mbps": "Throughput (MB/s)",
                "metrics.total_bytes": "Bytes Copied",
                "metrics.error_count": "Errors",
                "metrics.cpu_utilization_pct": "CPU %",
                "metrics.network_peak_mbps": "Net Peak (MB/s)",
                "metrics.avg_file_time_sec": "Avg File Time (s)",
                "metrics.min_file_size_bytes": "Min File Size (bytes)",
                "metrics.max_file_size_bytes": "Max File Size (bytes)",
                "metrics.avg_file_size_bytes": "Avg File Size (bytes)",
            }
            avail = {k: v for k, v in col_map.items() if k in worker_runs.columns}
            worker_display = worker_runs[list(avail.keys())].rename(columns=avail).copy()
            if "Worker" in worker_display.columns:
                worker_display["Worker"] = worker_display["Worker"].astype(int)
                worker_display = worker_display.sort_values("Worker")
            display(worker_display.reset_index(drop=True))